# lncRNA-DNAtriplex Example Usage

Links to open-source datasets:

[cBioPortal](https://www.cbioportal.org/datasets) \
[RefSeq](https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/) \
[LNCipedia](https://lncipedia.org/download)

## Dataset breakdown

In [22]:
!mkdir -p data

### cBioPortal
data_mutations.txt — cBioPortal somatic mutation data, MAF format. Each row is one mutation event in one patient: Hugo symbol, chromosome, start/end position, consequence, and variant classification. This is the source for mutated_dict.

This example uses the mutation data for uterine endometrial carcinoma (MSK 2024 cohort, ucec_msk_2024), a random example pulled from cBioPortal. From the cohort folder, data_mutations.txt is used.

In [8]:
%%bash

wget -P ./data https://datahub.assets.cbioportal.org/ucec_msk_2024.tar.gz
tar -xvzf ucec_msk_2024.tar.gz

--2026-03-25 22:08:29--  https://datahub.assets.cbioportal.org/ucec_msk_2024.tar.gz
Resolving datahub.assets.cbioportal.org (datahub.assets.cbioportal.org)... 108.138.64.115, 108.138.64.113, 108.138.64.70, ...
Connecting to datahub.assets.cbioportal.org (datahub.assets.cbioportal.org)|108.138.64.115|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 126898 (124K) [application/x-tar]
Saving to: ‘ucec_msk_2024.tar.gz’

ucec_msk_2024.tar.g 100%[===================>] 123.92K  --.-KB/s    in 0.02s   

2026-03-25 22:08:29 (6.91 MB/s) - ‘ucec_msk_2024.tar.gz’ saved [126898/126898]



### RefSeq
Healthy reference genome. Three necessary files used
1. genomic.fna.gz — The full human reference genome in FASTA format. One sequence record per chromosome/contig, identified by RefSeq accession IDs like NC_000001.11. This is what pyfaidx indexes to pull gene body sequences by coordinate.
2. genomic.gtf.gz — Gene annotation. Tab-delimited, one row per genomic feature (gene, transcript, exon, CDS, etc.). Gives you the chromosomal coordinates and strand for every gene by Hugo symbol — this is what _parse_gtf_for_genes() reads to know where to slice the FASTA.
3. genomic.assembly_report.txt — A mapping table that links human-readable chromosome names (1, chrX, MT) to their RefSeq accession IDs (NC_000001.11). Needed because the GTF uses one naming convention and the FASTA uses the other — load_chrom_map() bridges them.

*Make sure to use the most up to date NCBI_Build for RefSeq that matches data_mutations.txt. For this example, the MSK 2024 cohort data NCBI_Build = GRCh37, so download GRCh37.p13 accordingly*

In [ ]:
%%bash

wget  -P ./data https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/GCF_000001405.25_GRCh37.p13/GCF_000001405.25_GRCh37.p13_genomic.gtf.gz
wget  -P ./data https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/GCF_000001405.25_GRCh37.p13/GCF_000001405.25_GRCh37.p13_genomic.fna.gz
wget  -P ./data https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/GCF_000001405.25_GRCh37.p13/GCF_000001405.25_GRCh37.p13_assembly_report.txt

--2026-03-26 22:46:23--  https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/GCF_000001405.25_GRCh37.p13/GCF_000001405.25_GRCh37.p13_genomic.gtf.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.7, 130.14.250.31, 2607:f220:41e:250::13, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.7|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 25940824 (25M) [application/x-gzip]
Saving to: ‘GCF_000001405.25_GRCh37.p13_genomic.gtf.gz’

     0K .......... .......... .......... .......... ..........  0% 2.61M 9s
    50K .......... .......... .......... .......... ..........  0% 5.40M 7s
   100K .......... .......... .......... .......... ..........  0% 34.5M 5s
   150K .......... .......... .......... .......... ..........  0% 5.76M 5s
   200K .......... .......... .......... .......... ..........  0%  110M 4s
   250K .......... .......... .......... .......... ..........  1% 36.6M 3s
   300K .......... .......... ..........

### LNCipedia
lncipedia_5_2_hc.fasta — LNCipedia v5.2 high-confidence lncRNA sequences in FASTA format. The hc designation means these transcripts passed LNCipedia's quality filters (non-coding probability, multi-source validation). Each header is a transcript ID like lnc-TP53-1:1. This populates lncrna_dict.

In [ ]:
!wget  -P ./data https://lncipedia.org/downloads/lncipedia_5_2/full-database/lncipedia_5_2.fasta

## File Previews

In [42]:
def preview(f, nlines = 10):
    with open(f, 'r') as f:
        for i, line in enumerate(f, 1):
            if i > nlines:
                break
            print(line.strip())

In [43]:
main = Path.cwd()

# cBioPortal
mutationsF = [main / "ucec_msk_2024" / "data_mutations.txt"]

# RefSeq
genomeF = main / "GCF_000001405.25_GRCh37.p13_genomic.fna.gz"
annotationF = main / "GCF_000001405.25_GRCh37.p13_genomic.gtf.gz"
assemblyF = main / "GCF_000001405.25_GRCh37.p13_assembly_report.txt"

# LNCipedia
lncrnaF = main / "lncipedia_5_2_hc.fasta"

In [44]:
%%bash

ls ./ucec_msk_2024
echo
ls ./ucec_msk_2024/case_lists

~$Mapping.xlsx
case_lists
data_clinical_patient.txt
data_clinical_sample.txt
data_cna_hg19.seg
data_cna.txt
data_gene_panel_matrix.txt
data_mutations.txt
data_sv.txt
LICENSE
Mapping.xlsx
meta_clinical_patient.txt
meta_clinical_sample.txt
meta_cna_hg19_seg.txt
meta_cna.txt
meta_gene_panel_matrix.txt
meta_mutations.txt
meta_study.txt
meta_sv.txt
Readme.txt

cases_all.txt
cases_cnaseq.txt
cases_cna.txt
cases_sequenced.txt
cases_sv.txt


In [45]:
pd.set_option('display.max_columns', None)
df = pd.read_csv(mutationsF[0], sep = "\t")
df = df.dropna(axis=1, how='all') # remove all columns with just NaN
df.head(50)

,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Consequence,Variant_Classification,Variant_Type,Reference_Allele,Tumor_Seq_Allele1,Tumor_Seq_Allele2,dbSNP_RS,Tumor_Sample_Barcode,Validation_Status,Mutation_Status,Score,t_ref_count,t_alt_count,n_ref_count,n_alt_count,HGVSc,HGVSp,HGVSp_Short,Transcript_ID,RefSeq,Protein_position,Codons,IS_NEW
0,CTCF,10664,MSKCC,GRCh37,16,67645338,67645339,+,frameshift_variant,Frame_Shift_Ins,INS,-,-,A,rs1233586379,P-0023287-T01-IM6,Unknown,SOMATIC,MSK-IMPACT,551,40,564,0,ENST00000264010.4:c.610dup,p.Thr204AsnfsTer26,p.T204Nfs*26,ENST00000264010,NM_006565.3,201.0,-/A,NaN
1,PPP2R1A,5518,MSKCC,GRCh37,19,52715982,52715982,+,missense_variant,Missense_Mutation,SNP,C,C,T,rs1057519946,P-0023287-T01-IM6,Unknown,SOMATIC,MSK-IMPACT,676,103,748,0,ENST00000322088.6:c.547C>T,p.Arg183Trp,p.R183W,ENST00000322088,NM_014225.5,183.0,Cgg/Tgg,NaN
2,SOX17,64321,MSKCC,GRCh37,8,55372288,55372288,+,frameshift_variant,Frame_Shift_Del,DEL,C,C,-,rs1319573268,P-0023287-T01-IM6,Unknown,SOMATIC,MSK-IMPACT,660,83,726,1,ENST00000297316.4:c.983del,p.Pro328ArgfsTer59,p.P328Rfs*59,ENST00000297316,NM_022454.3,326.0,caC/ca,NaN
3,BARD1,580,MSKCC,GRCh37,2,215646085,215646085,+,frameshift_variant,Frame_Shift_Del,DEL,T,T,-,NaN,P-0023287-T01-IM6,Unknown,SOMATIC,MSK-IMPACT,507,65,861,1,ENST00000260947.4:c.513del,p.Asp172MetfsTer40,p.D172Mfs*40,ENST00000260947,NM_000465.2,171.0,aaA/aa,NaN
4,ARID1A,8289,MSKCC,GRCh37,1,27099103,27099103,+,frameshift_variant,Frame_Shift_Del,DEL,C,C,-,NaN,P-0023287-T01-IM6,Unknown,SOMATIC,MSK-IMPACT,711,85,892,2,ENST00000324856.7:c.3524del,p.Pro1175HisfsTer5,p.P1175Hfs*5,ENST00000324856,NM_006015.4,1173.0,atC/at,NaN
5,MST1R,4486,MSKCC,GRCh37,3,49936518,49936518,+,missense_variant,Missense_Mutation,SNP,C,C,T,rs867261592,P-0023287-T01-IM6,Unknown,SOMATIC,MSK-IMPACT,845,131,916,5,ENST00000296474.3:c.1409G>A,p.Arg470His,p.R470H,ENST00000296474,NM_002447.2,470.0,cGt/cAt,NaN
6,ARID1A,8289,MSKCC,GRCh37,1,27088681,27088682,+,frameshift_variant,Frame_Shift_Ins,INS,-,-,C,NaN,P-0023287-T01-IM6,Unknown,SOMATIC,MSK-IMPACT,896,134,944,0,ENST00000324856.7:c.2296dup,p.Gln766ProfsTer51,p.Q766Pfs*51,ENST00000324856,NM_006015.4,764.0,tcc/tCcc,NaN
7,PIK3R1,5295,MSKCC,GRCh37,5,67589618,67589618,+,stop_gained,Nonsense_Mutation,SNP,C,C,T,rs1057519838,P-0023287-T01-IM6,Unknown,SOMATIC,MSK-IMPACT,282,34,490,0,ENST00000274335.5:c.1381C>T,p.Arg461Ter,p.R461*,ENST00000274335,NaN,461.0,Cga/Tga,NaN
8,AGO2,27161,MSKCC,GRCh37,8,141542584,141542584,+,missense_variant,Missense_Mutation,SNP,G,G,A,rs1238499211,P-0023287-T01-IM6,Unknown,SOMATIC,MSK-IMPACT,706,124,814,1,ENST00000220592.5:c.2402C>T,p.Ala801Val,p.A801V,ENST00000220592,NM_012154.3,801.0,gCg/gTg,NaN
9,PLCG2,5336,MSKCC,GRCh37,16,81972512,81972512,+,missense_variant,Missense_Mutation,SNP,C,C,T,rs1263457484,P-0023287-T01-IM6,Unknown,SOMATIC,MSK-IMPACT,732,98,775,2,ENST00000359376.3:c.3305C>T,p.Thr1102Met,p.T1102M,ENST00000359376,NM_002661.3,1102.0,aCg/aTg,NaN


In [46]:
df.loc[:, ["Reference_Allele", "Tumor_Seq_Allele1", "Tumor_Seq_Allele2"]]

,Reference_Allele,Tumor_Seq_Allele1,Tumor_Seq_Allele2
0,-,-,A
1,C,C,T
2,C,C,-
3,T,T,-
4,C,C,-
...,...,...,...
1564,G,G,A
1565,G,G,A
1566,T,T,C
1567,G,G,A


## Pipeline

In [2]:
import pandas as pd
from Bio import SeqIO
from pyfaidx import Fasta
from pathlib import Path
import argparse
import subprocess
import os
from datetime import datetime
from typing import Dict, List

In [4]:
import sys
sys.path.insert(0, "./src")

from rules       import transfer_string, reverse_seq, complement
from stats       import calc_score
from sim         import SIM, Triplex, cluster_triplex, print_cluster
from data_loader import load_mutations, load_sequences, load_lncrnas, load_chrom_map
from alignment import Para, long_target, report

In [5]:
main = Path.cwd()

# cBioPortal
mutationsF = [main / "ucec_msk_2024" / "data_mutations.txt"]

# RefSeq
genomeF = main / "GCF_000001405.25_GRCh37.p13_genomic.fna.gz"
annotationF = main / "GCF_000001405.25_GRCh37.p13_genomic.gtf.gz"
assemblyF = main / "GCF_000001405.25_GRCh37.p13_assembly_report.txt"

# LNCipedia
lncrnaF = main / "lncipedia_5_2_hc.fasta"

In [6]:
args = argparse.Namespace(
    mutations = mutationsF,
    genome = genomeF,
    annotation = annotationF,
    lncrna = lncrnaF,
    assembly = assemblyF,
    outpath = "./results")

para = Para()

In [7]:
# ── ensure output directory exists ──────────────────────────────────
os.makedirs(args.outpath, exist_ok=True)
print(f"Output directory: {args.outpath}")

Output directory: ./results


In [8]:
# ── load dictionaries ─────────────────────────────────────────────────
print("Loading mutated gene data ...")
mutations_dict, gene_names = load_mutations(args.mutations)

print("Building chromosome map ...")
chrom_map = load_chrom_map(args.assembly)

print("Loading gene sequences ...")
healthy_seqs_dict, mutated_seqs_dict = load_sequences(
    fasta_path     = args.genome,
    gtf_path       = args.annotation,
    gene_names     = gene_names,
    chrom_map      = chrom_map,
    mutations_dict = mutations_dict,
)

print("Loading lncRNA sequences ...")
lncrna_seqs_dict = load_lncrnas(args.lncrna)

print(f"\n=== Data Summary ===")
print(f"  Genes with mutations:     {len(mutations_dict)}")
print(f"  Gene sequences retrieved: {len(healthy_seqs_dict)}")
print(f"  lncRNA transcripts:       {len(lncrna_seqs_dict)}")

Loading mutated gene data ...
[INFO] Loaded 381 unique mutated genes from 1 file(s).
Building chromosome map ...
[INFO] Loaded 594 chromosome mappings from GCF_000001405.25_GRCh37.p13_assembly_report.txt.
Loading gene sequences ...
[INFO] Recompressing FASTA with bgzip (this will take a while for large files)…
[INFO] Recompressed: /net/dali/home/mscbio/rop174/02604/lncRNA-DNAtriplex/GCF_000001405.25_GRCh37.p13_genomic.fna.gz
[INFO] Indexing genome FASTA (this may take a moment the first time)…
[INFO] Parsing GTF annotation…


[WARNING] IKBKE mut0: coordinates 206650027-206650027 outside gene interval 461330-487743, skipping.
[WARNING] IKBKE mut1: coordinates 206650020-206650020 outside gene interval 461330-487743, skipping.
[WARNING] IKBKE mut2: coordinates 206647725-206647725 outside gene interval 461330-487743, skipping.
[WARNING] HLA-A mut0: coordinates 29911082-29911082 outside gene interval 1203025-1206454, skipping.
[WARNING] HLA-B mut0: coordinates 31323363-31323363 outside gene interval 2701432-2704745, skipping.
[WARNING] HLA-B mut1: coordinates 31324220-31324220 outside gene interval 2701432-2704745, skipping.
[WARNING] HLA-B mut2: coordinates 31324132-31324132 outside gene interval 2701432-2704745, skipping.
[WARNING] STK19 mut0: coordinates 31948454-31948454 outside gene interval 3318796-3329068, skipping.
[WARNING] NOTCH4 mut0: coordinates 32180600-32180600 outside gene interval 3505441-3534689, skipping.
[WARNING] NOTCH4 mut1: coordinates 32170328-32170328 outside gene interval 3505441-3534689

[INFO] Retrieved reference sequences for 364 genes.
[INFO] Generated 1515/1539 mutated sequences.
Loading lncRNA sequences ...
[INFO] Loaded 107039 lncRNA transcripts.

=== Data Summary ===
  Genes with mutations:     381
  Gene sequences retrieved: 364
  lncRNA transcripts:       107039


In [9]:
# ── prepare gene/lncRNA lists ─────────────────────────────────────────
shared_genes = sorted(g for g in gene_names if g in healthy_seqs_dict)
lncrna_ids   = list(lncrna_seqs_dict.keys())

n_test  = 100
test_lncrnas = lncrna_ids[:n_test]

print(f"\n  Genes available for analysis: {len(shared_genes)}")
print(f"  lncRNAs to test:              {len(test_lncrnas)}")
print()


  Genes available for analysis: 364
  lncRNAs to test:              100



In [ ]:
# ── main loop ─────────────────────────────────────────────────────────
# results : {lnc_id: {gene: {"healthy": [Triplex], mut_label: [Triplex], ...}}}
loop_start = datetime.now()
print(f"Loop started at: {loop_start.strftime('%Y-%m-%d %H:%M:%S')}\n")

results:     Dict[str, Dict[str, Dict[str, List[Triplex]]]] = {}
total_pairs = 0
total_hits  = 0

for lnc_idx, lnc_id in enumerate(test_lncrnas):
    print("\nCurrent lnc:", lnc_id)
    lnc_seq = lncrna_seqs_dict[lnc_id]
    print(f"[{lnc_idx + 1}/{len(test_lncrnas)}] lncRNA: {lnc_id} "
          f"({len(lnc_seq)} nt)")
    results[lnc_id] = {}

    for gene in shared_genes:
        print("Current gene:", gene)
        healthy_seq  = healthy_seqs_dict[gene]
        gene_muts    = mutations_dict[gene]["mutations"]
        mut_seqs     = mutated_seqs_dict.get(gene, [])
        results[lnc_id][gene] = {}

        # ── healthy run ───────────────────────────────────────────────
        healthy_triplexes = long_target(para, lnc_seq, healthy_seq)
        results[lnc_id][gene]["healthy"] = healthy_triplexes
        total_hits += len(healthy_triplexes)

        # ── mutated runs ──────────────────────────────────────────────
        for mut_idx, (mut, mutated_seq) in enumerate(zip(gene_muts, mut_seqs)):
            if mutated_seq is None:
                continue
            mut_label = f"mut{mut_idx}_{mut.get('variant_class', 'unknown')}"
            mutated_triplexes = long_target(para, lnc_seq, mutated_seq)
            results[lnc_id][gene][mut_label] = mutated_triplexes
            total_hits += len(mutated_triplexes)

        total_pairs += 1

loop_end = datetime.now()
elapsed  = loop_end - loop_start

print(f"\nLoop ended at:   {loop_end.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Elapsed time:    {elapsed}")
print(f"Total (lncRNA × gene) pairs processed: {total_pairs}")
print(f"Total triplex hits across all runs:     {total_hits}")

Loop started at: 2026-03-27 01:55:07


Current lnc: LINC01725:44
[1/100] lncRNA: LINC01725:44 (1991 nt)
Current gene: ABL1
  DNA chunk @ 0 (5000 bp) ...


In [ ]:
# ── write report ──────────────────────────────────────────────────────
    report(results, mutations_dict, args.outpath)

In [7]:
# Debugging

# Check FASTA sequence IDs
from pyfaidx import Fasta
genome = Fasta(args.genome)
print(list(genome.keys())[:5])

# Check GTF chromosome names and gene attributes
import gzip
with gzip.open(args.annotation, "rt") as f:
    for i, line in enumerate(f):
        if not line.startswith("#"):
            print(line[:200])
            break

# Check what attribute keys exist for a gene feature
import gzip
with gzip.open(args.annotation, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) > 2 and parts[2] == "gene":
            print(parts[8])
            break

# Check a few gene names from your mutation data
print(list(gene_names)[:10])

from data_loader import _parse_gtf_attributes
import gzip

gtf_genes = set()
with gzip.open(args.annotation, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) > 2 and parts[2] == "gene":
            attrs = _parse_gtf_attributes(parts[8])
            sym = attrs.get("gene_name") or attrs.get("gene") or attrs.get("gene_id", "")
            gtf_genes.add(sym)

print(f"Total genes in GTF: {len(gtf_genes)}")
print(f"Overlap with mutation genes: {len(gene_names & gtf_genes)}")
print(f"Sample GTF genes: {list(gtf_genes)[:10]}")
print(f"Missing from GTF: {list(gene_names - gtf_genes)[:10]}")

# Also check the raw attribute string to make sure parsing is correct
import gzip
with gzip.open(args.annotation, "rt") as f:
    count = 0
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) > 2 and parts[2] == "gene":
            print(repr(parts[8]))
            count += 1
            if count == 3:
                break

# What chroms does the GTF produce for your genes?
import gzip
from data_loader import _parse_gtf_attributes

sample_chroms = set()
with gzip.open(args.annotation, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) > 2 and parts[2] == "gene":
            attrs = _parse_gtf_attributes(parts[8])
            sym = attrs.get("gene_name") or attrs.get("gene") or attrs.get("gene_id", "")
            if sym in gene_names:
                sample_chroms.add(parts[0])

print("GTF chroms for your genes:", list(sample_chroms)[:10])

# What keys does the FASTA actually have?
from pyfaidx import Fasta
genome = Fasta(args.genome)
print("FASTA keys:", list(genome.keys())[:10])

# What does chrom_map produce for one of these?
print("chrom_map sample:", {k: chrom_map[k] for k in list(sample_chroms)[:5] if k in chrom_map})

NC_000001.11	BestRefSeq	gene	11874	14409	.	+	.	gene_id "DDX11L1"; transcript_id ""; db_xref "GeneID:100287102"; db_xref "HGNC:HGNC:37102"; description "DEAD/H-box helicase 11 like 1 (pseudogene)"; gbk
